In [ ]:
import pandas as pd
import altair as alt
import mercury as mr
from IPython.display import display, Markdown

alt.renderers.enable("jupyter")
data = pd.read_csv("iphone_prices.csv")

In [ ]:
start_year = mr.Slider(label="From year", value=int(data.release_year.min()), min=int(data.release_year.min()), max=int(data.release_year.max()))
end_year = mr.Slider(label="Through year", value=int(data.release_year.max()), min=int(data.release_year.min()), max=int(data.release_year.max()))
status = mr.Select(label="Release status", choices=["Released", "All models", "Announced / unreleased"], value="Released")
price_type = mr.Select(label="Price measure", choices=["Full-device launch price", "Advertised launch price"])

In [ ]:
display(Markdown("# iPhone launch dashboard\nExplore launch prices, screen sizes, and storage across iPhone generations."))
display(Markdown(f"**Dataset snapshot:** {data['status_as_of'].max()} · {len(data)} models · Prices in USD"))

price_column = "price_usd" if price_type.value == "Full-device launch price" else "advertised_launch_price_usd"
filtered = data[data.release_year.between(start_year.value, end_year.value)].copy()
if status.value != "All models":
    status_code = "released" if status.value == "Released" else "announced_unreleased"
    filtered = filtered[filtered.release_status.eq(status_code)]
filtered = filtered.sort_values(["release_year", "model"])
priced = filtered.dropna(subset=[price_column]).copy()
if start_year.value > end_year.value:
    mr.Markdown("**Choose a From year no later than Through year.**")
elif filtered.empty:
    mr.Markdown("**No models match these filters.** Try a wider year range or All models.")

money = lambda value: f"${value:,.0f}" if pd.notna(value) else "—"
display(mr.Indicator([
    mr.Indicator(len(filtered), label="Models", variant="primary"),
    mr.Indicator(money(priced[price_column].median()), label="Median selected price", variant="teal"),
    mr.Indicator(money(priced[price_column].min()), label="Lowest selected price", variant="primary"),
    mr.Indicator(f"{filtered.screen_size_inches.mean():.1f} in" if not filtered.empty else "—", label="Average screen size", variant="teal"),
]))
mr.Markdown(f"**{price_type.value}** · {len(priced)} of {len(filtered)} selected models have a recorded price. Missing prices are excluded from price charts and summaries.")
if price_column == "advertised_launch_price_usd":
    mr.Markdown("Advertised prices may require carrier activation or a contract, and early models may show subsidized prices. They are not directly comparable to full-device prices.")
else:
    mr.Markdown("Recorded full-device prices use the CSV’s price basis: some require activation discounts or a contract; early entries may be launch-era prices. These are historical nominal prices, not current resale prices.")

In [ ]:
labels = {price_column: "Launch price (USD)", "release_year": "Release year", "screen_size_inches": "Screen size (inches)", "base_storage_gb": "Base storage (GB)", "release_status": "Release status", "model": "Model", "price_basis": "Price basis"}
if not priced.empty:
    color = alt.Color("release_status:N", title="Release status",
                      scale=alt.Scale(domain=["released", "announced_unreleased"], range=["#2563eb", "#f59e0b"]),
                      legend=alt.Legend(orient="top"))
    tooltip = [
        alt.Tooltip("model:N", title="Model"),
        alt.Tooltip("release_year:Q", title="Release year", format="d"),
        alt.Tooltip(f"{price_column}:Q", title=price_type.value, format="$,.0f"),
        alt.Tooltip("screen_size_inches:Q", title="Screen size (inches)"),
        alt.Tooltip("base_storage_gb:Q", title="Base storage (GB)"),
        alt.Tooltip("release_status:N", title="Release status"),
        alt.Tooltip("price_basis:N", title="Price basis"),
    ]
    price_axis = alt.Y(f"{price_column}:Q", title="Launch price (USD)", axis=alt.Axis(format="$,.0f"))
    # Keep edge points inside the plot, including single-year selections.
    year_min, year_max = priced.release_year.min(), priced.release_year.max()
    year_margin = max(1, (year_max - year_min) * 0.05)
    screen_min, screen_max = priced.screen_size_inches.min(), priced.screen_size_inches.max()
    screen_margin = max(0.2, (screen_max - screen_min) * 0.05)
    year_axis = alt.X("release_year:Q", title="Release year", scale=alt.Scale(zero=False, nice=False, domain=[year_min - year_margin, year_max + year_margin]), axis=alt.Axis(format="d", tickMinStep=1))
    base = alt.Chart(priced)

    mr.Markdown("## Launch prices over time")
    points = base.mark_circle(size=90, opacity=0.85).encode(x=year_axis, y=price_axis, color=color, tooltip=tooltip)
    yearly = priced.groupby("release_year", as_index=False)[price_column].median()
    median = alt.Chart(yearly).mark_line(color="#94a3b8", strokeDash=[4, 4]).encode(
        x=year_axis, y=price_axis,
        tooltip=[alt.Tooltip("release_year:Q", title="Release year", format="d"),
                 alt.Tooltip(f"{price_column}:Q", title="Yearly median", format="$,.0f")])
    price_chart = (median + points).properties(width=680, height=340).interactive().configure_view(stroke=None)
    display(price_chart)
    mr.Markdown("Dashed line: yearly median. Hover for model details; scroll to zoom or drag to pan.")

    mr.Markdown("## Screen size & price")
    screen_chart = base.mark_circle(size=110, opacity=0.8).encode(
        x=alt.X("screen_size_inches:Q", title="Screen size (inches)", scale=alt.Scale(zero=False, nice=False, domain=[screen_min - screen_margin, screen_max + screen_margin])),
        y=price_axis, color=color, tooltip=tooltip,
    ).properties(width=680, height=340).interactive().configure_view(stroke=None)
    display(screen_chart)
    mr.Markdown("Screen size uses the CSV’s main screen measurement; for foldables, inspect the inner and outer screen fields in the source details.")

In [ ]:
mr.Markdown("## Model explorer")
mr.Markdown("Search the table for a model. Download includes all original columns and source notes for the selected filters.")
columns = ["model", "release_year", price_column, "screen_size_inches", "base_storage_gb", "release_status", "price_basis"]
mr.Table(filtered[columns].rename(columns={**labels, "price_basis": "Price basis"}), search=True, page_size=12)
mr.Download(filtered.to_csv(index=False), filename="iphone_filtered.csv", label="Download filtered CSV", mime="text/csv", position="inline")
with mr.Expander("Source details & pricing notes"):
    mr.Markdown("Release status and source verification labels are provided by the CSV, as of its snapshot date. Blank full-device prices remain missing; advertised prices are never substituted. Chart medians weight each model equally.")
    details = ["model", "notes", "price_source_note", "price_verification_status", "price_source_url", "advertised_launch_price_source_url", "specification_source_url", "inner_screen_size_inches", "outer_screen_size_inches"]
    mr.Table(filtered[details].fillna(""), search=True, page_size=8)